In [1]:
import os

import pandas as pd
import matplotlib.pyplot as plt

from sqlalchemy import create_engine
from dotenv import load_dotenv
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
load_dotenv(PROJECT_ROOT / ".env")


db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME")


if db_password:
    db_url = (
        f"postgresql+psycopg2://"
        f"{db_user}:{db_password}"
        f"@{db_host}:{db_port}/{db_name}"
    )
else:
    db_url = (
        f"postgresql+psycopg2://"
        f"{db_user}@{db_host}:{db_port}/{db_name}"
    )


engine = create_engine(db_url)

In [2]:
query = """
SELECT *
FROM dbt.mart_scm_risk_weekly
WHERE is_complete_week = true
ORDER BY week_end_date
"""

risk = pd.read_sql(query, engine)

risk["week_end_date"] = pd.to_datetime(
    risk["week_end_date"]
)

print("shape:", risk.shape)
print(
    "period:",
    risk["week_end_date"].min(),
    "~",
    risk["week_end_date"].max()
)

risk.head()

shape: (155, 32)
period: 2023-10-22 00:00:00 ~ 2026-10-04 00:00:00


,week_end_date,product_id,product_name,calendar_days_in_week,is_complete_week,wholesale_qty,wholesale_qty_obs,wholesale_price,wholesale_price_obs,retail_price,...,jeju_sunshine_coverage_30d,jeju_precip_observed_30d,jeju_precip_coverage_30d,hapcheon_avg_temp_7d,hapcheon_avg_temp_30d,hapcheon_humidity_30d,hapcheon_sunshine_30d,hapcheon_sunshine_coverage_30d,hapcheon_precip_observed_30d,hapcheon_precip_coverage_30d
0,2023-10-22,P001,양파,7,True,5300.0,3,1517.000000,2,2260.0,...,1.0,41.6,0.433333,13.471429,17.136667,72.250000,6.200000,1.0,16.0,0.300000
1,2023-10-29,P001,양파,7,True,1800.0,1,1300.000000,1,2310.0,...,1.0,40.7,0.266667,13.928571,15.383333,71.726667,6.926667,1.0,10.6,0.233333
2,2023-11-05,P001,양파,7,True,4200.0,1,1200.000000,1,2310.0,...,1.0,36.3,0.233333,16.414286,15.206667,73.403333,6.790000,1.0,15.6,0.233333
3,2023-11-12,P001,양파,7,True,5250.0,2,1244.500000,2,1930.0,...,1.0,39.2,0.300000,10.357143,13.720000,71.006667,6.830000,1.0,41.7,0.266667
4,2023-11-19,P001,양파,7,True,6510.0,2,1255.072046,2,1980.0,...,1.0,69.1,0.333333,5.314286,11.476667,71.120000,6.573333,1.0,46.5,0.300000


In [3]:
risk[
    [
        "week_end_date",
        "wholesale_qty",
        "wholesale_price",
        "retail_price",
        "mokpo_avg_temp_30d",
        "jeju_avg_temp_30d",
        "hapcheon_avg_temp_30d"
    ]
].tail()

,week_end_date,wholesale_qty,wholesale_price,retail_price,mokpo_avg_temp_30d,jeju_avg_temp_30d,hapcheon_avg_temp_30d
150,2026-09-06,1425441.0,1146.088404,2120.0,27.800000,28.656667,26.510000
151,2026-09-13,1304881.6,1248.392878,1990.0,26.706667,28.056667,25.366667
152,2026-09-20,1419106.0,1081.654762,1930.0,25.783333,27.173333,24.413333
153,2026-09-27,787634.6,1252.059508,1620.0,24.323333,25.906667,22.776667
154,2026-10-04,967953.0,1239.519032,1990.0,22.686667,24.330000,21.026667


In [4]:
risk.isna().mean().sort_values(ascending=False)

wholesale_price                   0.096774
wholesale_qty                     0.090323
week_end_date                     0.000000
jeju_precip_coverage_30d          0.000000
jeju_avg_temp_30d                 0.000000
jeju_humidity_30d                 0.000000
jeju_sunshine_30d                 0.000000
jeju_sunshine_coverage_30d        0.000000
jeju_precip_observed_30d          0.000000
hapcheon_avg_temp_30d             0.000000
hapcheon_avg_temp_7d              0.000000
mokpo_precip_coverage_30d         0.000000
hapcheon_humidity_30d             0.000000
hapcheon_sunshine_30d             0.000000
hapcheon_sunshine_coverage_30d    0.000000
hapcheon_precip_observed_30d      0.000000
jeju_avg_temp_7d                  0.000000
mokpo_precip_observed_30d         0.000000
product_id                        0.000000
mokpo_sunshine_coverage_30d       0.000000
mokpo_sunshine_30d                0.000000
mokpo_humidity_30d                0.000000
mokpo_avg_temp_30d                0.000000
mokpo_avg_t

In [5]:
risk = risk.sort_values("week_end_date").copy()

risk["is_market_reliable"] = (
    (risk["wholesale_qty_obs"] >= 3) &
    (risk["wholesale_price_obs"] >= 3) &
    (risk["retail_price_obs"] >= 3)
)

risk[
    [
        "is_market_reliable",
        "wholesale_qty_obs",
        "wholesale_price_obs",
        "retail_price_obs"
    ]
].describe()

,wholesale_qty_obs,wholesale_price_obs,retail_price_obs
count,155.000000,155.000000,155.000000
mean,5.503226,5.341935,4.632258
std,2.311390,2.336716,0.738719
min,0.000000,0.000000,1.000000
25%,5.000000,5.000000,4.000000
50%,7.000000,6.000000,5.000000
75%,7.000000,7.000000,5.000000
max,7.000000,7.000000,5.000000


In [7]:
import numpy as np

risk["avg_daily_qty"] = (
    risk["wholesale_qty"]
    / risk["wholesale_qty_obs"].replace(0, np.nan)
)


# 거래량 최근 12주 baseline
risk["qty_baseline_12w"] = (
    risk["avg_daily_qty"]
    .shift(1)
    .rolling(
        window=12,
        min_periods=6
    )
    .median()
)

risk["qty_index_12w"] = (
    risk["avg_daily_qty"]
    / risk["qty_baseline_12w"]
)


# 도매가격 최근 12주 baseline
risk["price_baseline_12w"] = (
    risk["wholesale_price"]
    .shift(1)
    .rolling(
        window=12,
        min_periods=6
    )
    .median()
)

risk["price_index_12w"] = (
    risk["wholesale_price"]
    / risk["price_baseline_12w"]
)

In [8]:
risk[
    [
        "week_end_date",
        "avg_daily_qty",
        "qty_index_12w",
        "wholesale_price",
        "price_index_12w"
    ]
].tail(15)

,week_end_date,avg_daily_qty,qty_index_12w,wholesale_price,price_index_12w
140,2026-06-28,95884.857143,0.379794,768.033456,0.882548
141,2026-07-05,392732.342857,1.555587,925.301461,1.066115
142,2026-07-12,364330.285714,1.443089,943.000000,1.086507
143,2026-07-19,283351.500000,0.991211,852.800433,0.982580
144,2026-07-26,400124.900000,1.464036,995.927339,1.158288
145,2026-08-02,130380.500000,0.440604,1031.461402,1.199615
146,2026-08-09,190372.328571,0.696562,1096.598666,1.275371
147,2026-08-16,186558.428571,0.738946,1048.181402,1.205943
148,2026-08-23,153734.871429,0.640652,1178.433082,1.311697
149,2026-08-30,201124.714286,0.926391,1150.424127,1.231519


In [9]:
weather_cols = [
    "mokpo_avg_temp_30d",
    "mokpo_humidity_30d",
    "mokpo_sunshine_30d",

    "jeju_avg_temp_30d",
    "jeju_humidity_30d",
    "jeju_sunshine_30d",

    "hapcheon_avg_temp_30d",
    "hapcheon_humidity_30d",
    "hapcheon_sunshine_30d",
]

for col in weather_cols:

    baseline_col = f"{col}_baseline_12w"
    dev_col = f"{col}_dev_12w"

    risk[baseline_col] = (
        risk[col]
        .shift(1)
        .rolling(
            window=12,
            min_periods=6
        )
        .median()
    )

    risk[dev_col] = (
        risk[col]
        - risk[baseline_col]
    )

In [10]:
feature_cols = [
    "qty_index_12w",

    "mokpo_avg_temp_30d_dev_12w",
    "mokpo_humidity_30d_dev_12w",
    "mokpo_sunshine_30d_dev_12w",

    "jeju_avg_temp_30d_dev_12w",
    "jeju_humidity_30d_dev_12w",
    "jeju_sunshine_30d_dev_12w",

    "hapcheon_avg_temp_30d_dev_12w",
    "hapcheon_humidity_30d_dev_12w",
    "hapcheon_sunshine_30d_dev_12w",
]

current_analysis = risk[
    risk["is_market_reliable"]
].copy()

current_corr = (
    current_analysis[
        feature_cols + ["price_index_12w"]
    ]
    .corr(method="spearman")["price_index_12w"]
    .drop("price_index_12w")
    .sort_values()
)

current_corr

jeju_humidity_30d_dev_12w       -0.357699
mokpo_avg_temp_30d_dev_12w      -0.288706
hapcheon_avg_temp_30d_dev_12w   -0.275317
mokpo_humidity_30d_dev_12w      -0.203689
jeju_avg_temp_30d_dev_12w       -0.123885
hapcheon_humidity_30d_dev_12w   -0.099067
hapcheon_sunshine_30d_dev_12w   -0.098020
qty_index_12w                   -0.097344
jeju_sunshine_30d_dev_12w        0.036152
mokpo_sunshine_30d_dev_12w       0.097205
Name: price_index_12w, dtype: float64

In [11]:
risk["next_price_index"] = (
    risk["price_index_12w"]
    .shift(-1)
)

risk["next_market_reliable"] = (
    risk["is_market_reliable"]
    .shift(-1)
    .fillna(False)
    .astype(bool)
)

/var/folders/9k/gfjrzlbs4m944f17wkmd4j600000gn/T/ipykernel_43604/3652180196.py:9: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)


In [12]:
next_analysis = risk[
    risk["is_market_reliable"] &
    risk["next_market_reliable"]
].copy()

next_corr = (
    next_analysis[
        feature_cols + ["next_price_index"]
    ]
    .corr(method="spearman")["next_price_index"]
    .drop("next_price_index")
    .sort_values()
)

next_corr

jeju_humidity_30d_dev_12w       -0.295803
mokpo_avg_temp_30d_dev_12w      -0.293855
hapcheon_avg_temp_30d_dev_12w   -0.291224
hapcheon_sunshine_30d_dev_12w   -0.151544
mokpo_humidity_30d_dev_12w      -0.128131
qty_index_12w                   -0.090558
jeju_avg_temp_30d_dev_12w       -0.082781
hapcheon_humidity_30d_dev_12w   -0.073948
jeju_sunshine_30d_dev_12w        0.014334
mokpo_sunshine_30d_dev_12w       0.043086
Name: next_price_index, dtype: float64

In [13]:
print("전체 complete weeks:", len(risk))
print("현재가격 분석 weeks:", len(current_analysis))
print("다음주가격 분석 weeks:", len(next_analysis))

전체 complete weeks: 155
현재가격 분석 weeks: 125
다음주가격 분석 weeks: 117
